# 🛍️ Product Review Sentiment Analysis
**Platforms supported:** Amazon.in · Flipkart  
**Sentiment engine:** VADER (optimized for short, informal product reviews)  
**Storage:** SQLite — scraping progress is saved as you go, safe to resume after interruption

---
> ⚠️ **Legal Notice:** This notebook is for **educational purposes only.**  
> Scraping Amazon and Flipkart may violate their Terms of Service.  
> Do not use for commercial purposes or large-scale data collection.  
> Always check a site's `robots.txt` and ToS before scraping.

## Cell 1 — Install Dependencies

In [ ]:
# Run once to install everything needed
import subprocess, sys
packages = [
    'selenium', 'webdriver-manager', 'beautifulsoup4',
    'lxml', 'vaderSentiment', 'pandas', 'numpy',
    'matplotlib', 'seaborn', 'wordcloud'
]
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '--quiet'] + packages)
print('✅ All packages installed.')

## Cell 2 — Imports & Global Config

In [ ]:
import re
import time
import random
import hashlib
import logging
import sqlite3
import warnings
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from wordcloud import WordCloud

from bs4 import BeautifulSoup
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from webdriver_manager.chrome import ChromeDriverManager

from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer

warnings.filterwarnings('ignore')
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (14, 6)

# Logging: errors go to scraper.log, warnings printed to console
logging.basicConfig(
    filename='scraper.log',
    level=logging.WARNING,
    format='%(asctime)s %(levelname)s %(message)s'
)

# VADER analyzer — shared instance, no need to re-init per review
vader = SentimentIntensityAnalyzer()

# SQLite DB for persistent storage
DB_PATH = 'reviews.db'

print('✅ Imports complete. VADER ready.')

## Cell 3 — Database Setup
Reviews are written to SQLite as they're scraped.  
If a run crashes, you won't lose data — just rerun and it resumes.

In [ ]:
def init_db(db_path: str = DB_PATH) -> sqlite3.Connection:
    """Create the reviews table if it doesn't already exist."""
    conn = sqlite3.connect(db_path)
    conn.execute('''
        CREATE TABLE IF NOT EXISTS reviews (
            review_hash   TEXT PRIMARY KEY,
            review_id     TEXT,
            platform      TEXT,
            product_name  TEXT,
            rating        REAL,
            review_title  TEXT,
            review_text   TEXT,
            reviewer_name TEXT,
            review_date   TEXT,
            verified      INTEGER
        )
    ''')
    conn.commit()
    return conn


def make_hash(reviewer: str, text: str) -> str:
    """Fingerprint a review to detect duplicates across pages."""
    return hashlib.md5(f"{reviewer}{text[:60]}".encode()).hexdigest()


def save_review(conn: sqlite3.Connection, platform: str, review_id: str,
                product_name: str, rating, title: str, text: str,
                reviewer: str, date: str, verified: bool) -> bool:
    """
    Insert a review into the DB. Returns True if inserted, False if duplicate.
    Uses INSERT OR IGNORE so duplicates are silently skipped.
    """
    h = make_hash(reviewer, text)
    try:
        conn.execute(
            '''INSERT OR IGNORE INTO reviews VALUES (?,?,?,?,?,?,?,?,?,?)''',
            (h, review_id, platform, product_name, rating,
             title, text, reviewer, date, int(verified))
        )
        conn.commit()
        return conn.total_changes > 0
    except sqlite3.Error as e:
        logging.error(f"DB insert error: {e}")
        return False


def load_reviews(db_path: str = DB_PATH) -> pd.DataFrame:
    """Load all saved reviews into a DataFrame."""
    conn = sqlite3.connect(db_path)
    df = pd.read_sql('SELECT * FROM reviews', conn)
    conn.close()
    return df


print('✅ Database helpers ready.')

## Cell 4 — Browser Utilities
Single shared driver builder + a smart delay helper that mimics human pacing.

In [ ]:
# A pool of real user-agents to rotate between requests
USER_AGENTS = [
    'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/124.0.0.0 Safari/537.36',
    'Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/123.0.0.0 Safari/537.36',
    'Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/122.0.0.0 Safari/537.36',
    'Mozilla/5.0 (Windows NT 10.0; Win64; x64; rv:124.0) Gecko/20100101 Firefox/124.0',
]


def build_driver(headless: bool = True) -> webdriver.Chrome:
    """
    Build a Chrome driver with anti-detection options.
    Set headless=False to watch the browser (useful for debugging CAPTCHAs).
    """
    opts = Options()
    if headless:
        opts.add_argument('--headless=new')          # new headless mode (Chrome 112+)
    opts.add_argument('--no-sandbox')
    opts.add_argument('--disable-dev-shm-usage')
    opts.add_argument('--disable-blink-features=AutomationControlled')
    opts.add_argument(f'--user-agent={random.choice(USER_AGENTS)}')
    opts.add_experimental_option('excludeSwitches', ['enable-automation'])
    opts.add_experimental_option('useAutomationExtension', False)
    opts.add_argument('--window-size=1920,1080')

    driver = webdriver.Chrome(
        service=Service(ChromeDriverManager().install()),
        options=opts
    )
    # Mask navigator.webdriver fingerprint
    driver.execute_cdp_cmd(
        'Page.addScriptToEvaluateOnNewDocument',
        {'source': 'Object.defineProperty(navigator, "webdriver", {get: () => undefined})'}
    )
    return driver


def smart_delay(base: float = 2.0, jitter: float = 2.0, backoff: float = 1.0):
    """
    Human-paced delay with optional backoff multiplier.
    Increase backoff when retrying after a block or error.
    """
    delay = (base + random.uniform(0, jitter)) * backoff
    time.sleep(delay)


print('✅ Browser utilities ready.')

## Cell 5 — Amazon Scraper
Works from either a product page URL or a direct reviews page URL.  
Each review is written to SQLite immediately — no data lost on crash.

In [ ]:
def _parse_amazon_reviews(soup: BeautifulSoup, product_name: str,
                           conn: sqlite3.Connection, page: int) -> int:
    """Parse all review blocks from a BeautifulSoup page and save to DB."""
    review_blocks = soup.find_all('div', {'data-hook': 'review'})
    saved = 0

    for idx, block in enumerate(review_blocks):
        try:
            # --- Rating ---
            rating = None
            rating_elem = block.find('i', {'data-hook': 'review-star-rating'}) or \
                          block.find('i', {'data-hook': 'cmps-review-star-rating'})
            if rating_elem:
                try:
                    rating = float(rating_elem.text.strip().split()[0])
                except ValueError as e:
                    logging.warning(f"AMZ page {page} idx {idx} — rating parse failed: {e}")

            # --- Title ---
            title_elem = block.find('a', {'data-hook': 'review-title'}) or \
                         block.find('span', {'data-hook': 'review-title'})
            title = title_elem.get_text(strip=True) if title_elem else ''

            # --- Body ---
            text_elem = block.find('span', {'data-hook': 'review-body'})
            text = text_elem.get_text(strip=True) if text_elem else ''

            # Skip reviews with no meaningful body
            if not text or len(text) < 10:
                continue

            # --- Reviewer ---
            name_elem = block.find('span', class_='a-profile-name')
            reviewer = name_elem.get_text(strip=True) if name_elem else 'Anonymous'

            # --- Date ---
            date_elem = block.find('span', {'data-hook': 'review-date'})
            date = date_elem.get_text(strip=True) if date_elem else ''

            # --- Verified ---
            verified = bool(block.find('span', {'data-hook': 'avp-badge'}))

            inserted = save_review(
                conn, 'amazon', f'AMZ_{page}_{idx}', product_name,
                rating, title, text, reviewer, date, verified
            )
            if inserted:
                saved += 1

        except Exception as e:
            logging.warning(f"AMZ page {page} idx {idx} — skipped review: {e}")
            continue

    return saved


def scrape_amazon(url: str, max_pages: int = 5, headless: bool = True) -> int:
    """
    Scrape Amazon reviews from a product page OR a direct reviews page URL.
    Reviews are saved to SQLite as they go. Returns total reviews saved.

    Parameters
    ----------
    url        : Amazon product URL (amazon.in/dp/...) or reviews URL (amazon.in/product-reviews/...)
    max_pages  : How many review pages to scrape
    headless   : Run Chrome headlessly. Set False to watch browser / solve CAPTCHAs manually.
    """
    print('=' * 60)
    print('🛒  AMAZON SCRAPER')
    print('=' * 60)

    conn = init_db()
    driver = build_driver(headless=headless)
    total_saved = 0
    backoff = 1.0

    try:
        print(f'📄 Loading: {url}')
        driver.get(url)
        smart_delay(3, 2)

        # ── If we landed on a product page, navigate to the reviews page ──
        if 'product-reviews' not in url:
            product_name = 'Unknown Product'
            try:
                product_name = driver.find_element(By.ID, 'productTitle').text.strip()
                print(f'📦 Product: {product_name}')
            except Exception as e:
                logging.warning(f'Could not get product title: {e}')

            # Scroll halfway to trigger lazy-loaded review section
            driver.execute_script('window.scrollTo(0, document.body.scrollHeight / 2);')
            smart_delay(2, 1)

            reviews_url = None
            selectors = [
                (By.CSS_SELECTOR, "a[data-hook='see-all-reviews-link-foot']"),
                (By.CSS_SELECTOR, "a[data-hook='see-all-reviews-link-head']"),
                (By.XPATH, "//a[contains(@href, 'product-reviews')]"),
            ]
            for by, sel in selectors:
                try:
                    elem = driver.find_element(by, sel)
                    reviews_url = elem.get_attribute('href')
                    if reviews_url:
                        break
                except Exception:
                    continue

            if reviews_url:
                print(f'🔗 Navigating to reviews page...')
                driver.get(reviews_url)
                smart_delay(3, 2)
            else:
                print('⚠️  Could not find reviews page link — trying to extract from product page directly.')
        else:
            # We're already on the reviews page
            product_name = 'Unknown Product'
            try:
                product_name = driver.find_element(
                    By.CSS_SELECTOR, "a[data-hook='product-link']"
                ).text.strip()
                print(f'📦 Product: {product_name}')
            except Exception as e:
                logging.warning(f'Could not get product title from reviews page: {e}')

        # ── Paginate through reviews ──
        for page in range(max_pages):
            print(f'\n📖 Page {page + 1}/{max_pages}...')

            # Wait for at least one review div to appear
            try:
                WebDriverWait(driver, 10).until(
                    EC.presence_of_element_located((By.CSS_SELECTOR, "div[data-hook='review']"))
                )
                backoff = 1.0   # reset backoff on success
            except Exception:
                print(f'   ⚠️  Reviews not found on page {page + 1} — possible CAPTCHA or block.')
                print(f'   💡 Tip: Re-run with headless=False to solve CAPTCHA manually.')
                backoff = min(backoff * 2, 8.0)  # exponential backoff, cap at 8×
                smart_delay(4, 2, backoff)
                if page == 0:
                    break   # no point continuing if page 1 is blocked

            # Scroll to bottom to trigger any lazy-loaded reviews
            driver.execute_script('window.scrollTo(0, document.body.scrollHeight);')
            smart_delay(1, 0.5)

            soup = BeautifulSoup(driver.page_source, 'lxml')
            saved_this_page = _parse_amazon_reviews(soup, product_name, conn, page)
            total_saved += saved_this_page
            print(f'   ✓ {saved_this_page} new reviews saved (total: {total_saved})')

            # Go to next page
            if page < max_pages - 1:
                try:
                    next_btn = driver.find_element(By.CSS_SELECTOR, 'li.a-last a')
                    driver.execute_script('arguments[0].click();', next_btn)
                    smart_delay(2, 2, backoff)
                except Exception:
                    print('   ℹ️  No more pages.')
                    break

    except Exception as e:
        print(f'❌ Scraper error: {e}')
        logging.error(f'Amazon scraper fatal error: {e}')
    finally:
        driver.quit()
        conn.close()

    print(f'\n✅ Amazon scrape complete — {total_saved} reviews saved to {DB_PATH}')
    return total_saved


print('✅ Amazon scraper ready.')

## Cell 6 — Flipkart Scraper

In [ ]:
# Flipkart uses obfuscated CSS class names that change on every UI deploy.
# We use data-driven fallbacks: look for structural patterns rather than
# specific class strings wherever possible.

def _parse_flipkart_reviews(soup: BeautifulSoup, product_name: str,
                             conn: sqlite3.Connection, page: int) -> int:
    """Parse Flipkart review cards and save to DB."""

    # Flipkart review cards sit inside <div class="col EPCmJX Ma-XX5"> or similar.
    # The only stable attribute is that each card contains a star rating div
    # and a review body. We find by content, not class name.
    review_blocks = soup.find_all('div', attrs={'class': lambda c: c and 'EPCmJX' in c})

    # Broad fallback: any div that contains both a rating and review text
    if not review_blocks:
        candidates = soup.find_all('div')
        review_blocks = [
            d for d in candidates
            if d.find('div', attrs={'class': lambda c: c and 'XQDdHH' in c}) and
               d.find('div', attrs={'class': lambda c: c and 'ZmyHeo' in c})
        ]

    saved = 0
    for idx, block in enumerate(review_blocks):
        try:
            # --- Rating ---
            rating = None
            # Try known class fragments; Flipkart rating divs contain a single digit
            for cls_fragment in ['XQDdHH', 'hGSR34', 'iB9i3r']:
                rating_elem = block.find('div', attrs={'class': lambda c: c and cls_fragment in c})
                if rating_elem:
                    try:
                        rating = float(rating_elem.get_text(strip=True).split()[0])
                        break
                    except ValueError:
                        pass

            # --- Title ---
            title = ''
            for cls_fragment in ['z9E0IG', '_2-N8zT']:
                title_elem = block.find('p', attrs={'class': lambda c: c and cls_fragment in c})
                if title_elem:
                    title = title_elem.get_text(strip=True)
                    break

            # --- Body ---
            text = ''
            for cls_fragment in ['ZmyHeo', 't-ZTKy', 'row text']:
                text_elem = block.find('div', attrs={'class': lambda c: c and cls_fragment in c})
                if text_elem:
                    text = text_elem.get_text(strip=True)
                    break

            if not text or len(text) < 10:
                continue

            # --- Reviewer + Date ---
            # Both often live in the same element: "Username, Month Year"
            reviewer, date = 'Anonymous', ''
            for cls_fragment in ['_2sc7ZR', 'row nameAge']:
                meta_elem = block.find('p', attrs={'class': lambda c: c and cls_fragment in c})
                if meta_elem:
                    parts = meta_elem.get_text(strip=True).split(',')
                    reviewer = parts[0].strip()
                    date = parts[-1].strip() if len(parts) > 1 else ''
                    break

            # --- Verified ---
            verified = bool(block.find(
                string=lambda s: s and 'certified buyer' in s.lower()
            ))

            inserted = save_review(
                conn, 'flipkart', f'FK_{page}_{idx}', product_name,
                rating, title, text, reviewer, date, verified
            )
            if inserted:
                saved += 1

        except Exception as e:
            logging.warning(f'FK page {page} idx {idx} — skipped: {e}')
            continue

    return saved


def scrape_flipkart(url: str, max_pages: int = 5, headless: bool = True) -> int:
    """
    Scrape Flipkart product reviews.
    Reviews are saved to SQLite as they go. Returns total reviews saved.

    Parameters
    ----------
    url        : Flipkart product page URL
    max_pages  : How many review pages to scrape
    headless   : Run Chrome headlessly.
    """
    print('=' * 60)
    print('🛍️  FLIPKART SCRAPER')
    print('=' * 60)

    conn = init_db()
    driver = build_driver(headless=headless)
    total_saved = 0

    try:
        print(f'📄 Loading: {url}')
        driver.get(url)
        smart_delay(3, 2)

        # ── Get product name ──
        product_name = 'Unknown Product'
        for css in ['span.VU-ZEz', 'span.B_NuCI', 'h1.yhB1nd']:
            try:
                product_name = driver.find_element(By.CSS_SELECTOR, css).text.strip()
                if product_name:
                    break
            except Exception:
                continue
        print(f'📦 Product: {product_name}')

        # ── Navigate to all-reviews page ──
        driver.execute_script('window.scrollTo(0, document.body.scrollHeight / 2);')
        smart_delay(1, 1)

        try:
            all_reviews_link = driver.find_element(
                By.XPATH,
                "//span[contains(text(),'All') and contains(text(),'review')] | "
                "//div[contains(text(),'All') and contains(text(),'review')]"
            )
            driver.execute_script('arguments[0].click();', all_reviews_link)
            smart_delay(3, 1)
            print('🔗 Navigated to all reviews page.')
        except Exception:
            print('⚠️  Could not find "All reviews" link — scraping from product page directly.')

        # ── Paginate ──
        for page in range(max_pages):
            print(f'\n📖 Page {page + 1}/{max_pages}...')

            driver.execute_script('window.scrollTo(0, document.body.scrollHeight);')
            smart_delay(2, 1)

            soup = BeautifulSoup(driver.page_source, 'lxml')
            saved_this_page = _parse_flipkart_reviews(soup, product_name, conn, page)
            total_saved += saved_this_page
            print(f'   ✓ {saved_this_page} new reviews saved (total: {total_saved})')

            if saved_this_page == 0 and page > 0:
                print('   ℹ️  No new reviews found — stopping early.')
                break

            if page < max_pages - 1:
                try:
                    next_btn = driver.find_element(
                        By.XPATH,
                        "//a[contains(@class,'_9QVEpD')] | //span[text()='Next']/parent::a"
                    )
                    driver.execute_script('arguments[0].click();', next_btn)
                    smart_delay(2, 2)
                except Exception:
                    print('   ℹ️  No more pages.')
                    break

    except Exception as e:
        print(f'❌ Scraper error: {e}')
        logging.error(f'Flipkart scraper fatal error: {e}')
    finally:
        driver.quit()
        conn.close()

    print(f'\n✅ Flipkart scrape complete — {total_saved} reviews saved to {DB_PATH}')
    return total_saved


print('✅ Flipkart scraper ready.')

## Cell 7 — Text Cleaning & VADER Sentiment
Key improvements over the original:
- Numbers are **kept** (`5 star`, `2 weeks`, `30 mins` now retain meaning)
- VADER is used instead of TextBlob — designed for short, informal text

In [ ]:
def clean_text(text: str) -> str:
    """
    Lightweight text normalisation.
    Keeps numbers (5 star, 2 weeks) — unlike the original which stripped them.
    """
    if pd.isna(text):
        return ''
    text = str(text).lower()
    text = re.sub(r'[^a-z0-9\s]', ' ', text)   # keep letters AND digits
    text = re.sub(r'\s+', ' ', text).strip()
    return text


def get_sentiment(text: str) -> pd.Series:
    """
    Score sentiment with VADER.
    Returns (compound, pos, neu, neg, label) as a Series.

    VADER compound score thresholds:
      >= +0.05  → Positive
      <= -0.05  → Negative
      else      → Neutral
    """
    if not text or len(text) < 3:
        return pd.Series([0.0, 0.0, 1.0, 0.0, 'Neutral'],
                         index=['compound', 'pos', 'neu', 'neg', 'sentiment'])

    scores = vader.polarity_scores(text)
    compound = scores['compound']

    if compound >= 0.05:
        label = 'Positive'
    elif compound <= -0.05:
        label = 'Negative'
    else:
        label = 'Neutral'

    return pd.Series(
        [compound, scores['pos'], scores['neu'], scores['neg'], label],
        index=['compound', 'pos', 'neu', 'neg', 'sentiment']
    )


def perform_sentiment_analysis(df: pd.DataFrame) -> pd.DataFrame:
    """Clean text and run VADER sentiment on every review in the DataFrame."""
    print('🧹 Cleaning text...')
    df = df.copy()
    df['cleaned_text'] = df['review_text'].apply(clean_text)

    print('🤖 Running VADER sentiment analysis...')
    sentiment_df = df['cleaned_text'].apply(get_sentiment)
    df = pd.concat([df, sentiment_df], axis=1)

    print(f'✅ Done — {len(df)} reviews analysed.')
    return df


print('✅ Sentiment functions ready.')

## Cell 8 — Visualisations

In [ ]:
COLORS = {'Positive': '#2ecc71', 'Negative': '#e74c3c', 'Neutral': '#95a5a6'}


def visualize_sentiment(df: pd.DataFrame):
    """Four-panel overview + word clouds."""

    sentiment_counts = df['sentiment'].value_counts().reindex(
        ['Positive', 'Neutral', 'Negative'], fill_value=0
    )
    colors = [COLORS[s] for s in sentiment_counts.index]

    # ── Panel 1: Overview ──
    fig, axes = plt.subplots(2, 2, figsize=(16, 12))
    fig.suptitle(
        f"Sentiment Analysis — {df['product_name'].iloc[0]}",
        fontsize=16, fontweight='bold', y=1.01
    )

    # Pie
    sentiment_counts.plot(kind='pie', ax=axes[0, 0], autopct='%1.1f%%',
                          colors=colors, startangle=90)
    axes[0, 0].set_title('Sentiment Distribution', fontweight='bold')
    axes[0, 0].set_ylabel('')

    # Bar
    sentiment_counts.plot(kind='bar', ax=axes[0, 1], color=colors)
    axes[0, 1].set_title('Sentiment Frequency', fontweight='bold')
    axes[0, 1].set_xlabel('Sentiment')
    axes[0, 1].set_ylabel('Count')
    axes[0, 1].tick_params(axis='x', rotation=0)
    for bar in axes[0, 1].patches:
        axes[0, 1].text(
            bar.get_x() + bar.get_width() / 2,
            bar.get_height() + 0.3,
            str(int(bar.get_height())),
            ha='center', va='bottom', fontsize=11
        )

    # Compound score distribution
    axes[1, 0].hist(df['compound'], bins=25, color='steelblue', edgecolor='white')
    axes[1, 0].axvline(x=0.05, color='green', linestyle='--', label='Positive threshold')
    axes[1, 0].axvline(x=-0.05, color='red', linestyle='--', label='Negative threshold')
    axes[1, 0].set_title('VADER Compound Score Distribution', fontweight='bold')
    axes[1, 0].set_xlabel('Compound Score')
    axes[1, 0].set_ylabel('Frequency')
    axes[1, 0].legend(fontsize=9)

    # Rating vs Sentiment
    if 'rating' in df.columns and df['rating'].notna().any():
        ct = pd.crosstab(df['rating'], df['sentiment'])
        for col in ['Positive', 'Neutral', 'Negative']:
            if col not in ct.columns:
                ct[col] = 0
        ct[['Positive', 'Neutral', 'Negative']].plot(
            kind='bar', ax=axes[1, 1],
            color=[COLORS['Positive'], COLORS['Neutral'], COLORS['Negative']]
        )
        axes[1, 1].set_title('Sentiment by Star Rating', fontweight='bold')
        axes[1, 1].set_xlabel('Rating')
        axes[1, 1].set_ylabel('Count')
        axes[1, 1].tick_params(axis='x', rotation=0)
        axes[1, 1].legend(title='Sentiment')
    else:
        axes[1, 1].text(0.5, 0.5, 'No rating data', ha='center', va='center', fontsize=12)
        axes[1, 1].axis('off')

    plt.tight_layout()
    plt.show()

    # ── Panel 2: Word clouds ──
    print('\n📝 Generating word clouds...')
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))

    for ax, sentiment in zip(axes, ['Positive', 'Negative', 'Neutral']):
        subset = df[df['sentiment'] == sentiment]['cleaned_text']
        combined = ' '.join(subset)
        if combined.strip():
            wc = WordCloud(
                width=500, height=350,
                background_color='white',
                colormap='RdYlGn' if sentiment != 'Negative' else 'Reds',
                max_words=60
            ).generate(combined)
            ax.imshow(wc, interpolation='bilinear')
            ax.set_title(f'{sentiment} Reviews ({len(subset)})',
                         fontsize=13, fontweight='bold',
                         color=COLORS[sentiment])
        else:
            ax.text(0.5, 0.5, f'No {sentiment} reviews', ha='center', va='center')
        ax.axis('off')

    plt.tight_layout()
    plt.show()


print('✅ Visualisation functions ready.')

## Cell 9 — Summary Report

In [ ]:
STOP_WORDS = {
    'the','a','an','and','or','but','in','on','at','to','for','of','with',
    'is','it','this','that','very','so','my','i','im','ive','dont','didnt',
    'was','were','have','has','its','be','are','not','just','really','also',
    'will','can','get','got','one','product','phone'
}


def top_keywords(df: pd.DataFrame, sentiment: str, n: int = 6) -> list:
    text = ' '.join(df[df['sentiment'] == sentiment]['cleaned_text'])
    words = [w for w in text.split() if w not in STOP_WORDS and len(w) > 2]
    return Counter(words).most_common(n)


def generate_summary_report(df: pd.DataFrame, save_csv: bool = True):
    """Print a structured summary and optionally export results to CSV."""

    product_name = df['product_name'].iloc[0] if not df.empty else 'Unknown'
    total = len(df)
    counts = df['sentiment'].value_counts()

    print('\n' + '═' * 60)
    print(f'  📦  {product_name}')
    print(f'  📊  {total} reviews analysed')
    print('═' * 60)

    print('\n🎯 SENTIMENT BREAKDOWN')
    for s, emoji in [('Positive', '😊'), ('Neutral', '😐'), ('Negative', '😞')]:
        c = counts.get(s, 0)
        pct = c / total * 100 if total else 0
        bar = '█' * int(pct / 5)
        print(f'   {emoji} {s:<10} {c:>4}  ({pct:5.1f}%)  {bar}')

    print('\n📈 AVERAGE SCORES (VADER)')
    print(f'   Compound : {df["compound"].mean():+.3f}  (−1 very negative → +1 very positive)')
    print(f'   Positive : {df["pos"].mean():.3f}')
    print(f'   Negative : {df["neg"].mean():.3f}')
    if df['rating'].notna().any():
        print(f'   ⭐ Avg rating : {df["rating"].mean():.2f} / 5.0')

    if 'verified' in df.columns:
        v = df['verified'].sum()
        print(f'\n✅ Verified purchases: {v} ({v/total*100:.1f}%)')

    print('\n📝 TOP KEYWORDS')
    for s, emoji in [('Positive', '😊'), ('Negative', '😞')]:
        if s in df['sentiment'].values:
            kws = top_keywords(df, s)
            print(f'   {emoji} {s}: {", ".join(f"{w}({c})" for w, c in kws)}')

    # Overall verdict
    pos_pct = counts.get('Positive', 0) / total * 100 if total else 0
    print('\n' + '─' * 60)
    if pos_pct >= 70:
        verdict = '⭐⭐⭐⭐⭐  HIGHLY RECOMMENDED'
    elif pos_pct >= 50:
        verdict = '⭐⭐⭐⭐    RECOMMENDED'
    elif pos_pct >= 30:
        verdict = '⭐⭐⭐      MIXED REVIEWS'
    else:
        verdict = '⭐⭐        CAUTION ADVISED'
    print(f'  💡 {verdict}')
    print('═' * 60)

    if save_csv:
        out_cols = ['review_id', 'platform', 'product_name', 'rating',
                    'review_title', 'review_text', 'reviewer_name',
                    'review_date', 'verified', 'compound', 'pos', 'neu', 'neg', 'sentiment']
        out_cols = [c for c in out_cols if c in df.columns]
        fname = 'sentiment_results.csv'
        df[out_cols].to_csv(fname, index=False)
        print(f'\n💾 Results saved → {fname}')


print('✅ Report functions ready.')

## Cell 10 — Main Entry Point
Edit the URL below and run. That's it.

In [ ]:
# ─────────────────────────────────────────────────────────────
#  CONFIGURE HERE
# ─────────────────────────────────────────────────────────────

PRODUCT_URL = "YOUR_PRODUCT_URL_HERE"   # Amazon or Flipkart URL
MAX_PAGES   = 5                         # Number of review pages to scrape
HEADLESS    = True                      # False = visible browser (useful if blocked)

# ─────────────────────────────────────────────────────────────

def run_analysis(url: str, max_pages: int = 5, headless: bool = True) -> pd.DataFrame:
    """
    Full pipeline: scrape → analyse → visualise → report.
    Returns the final DataFrame with sentiment scores.
    """
    if 'amazon' in url.lower():
        scrape_amazon(url, max_pages=max_pages, headless=headless)
    elif 'flipkart' in url.lower():
        scrape_flipkart(url, max_pages=max_pages, headless=headless)
    else:
        print('❌ URL must be from Amazon (amazon.in) or Flipkart (flipkart.com)')
        return pd.DataFrame()

    # Load all scraped reviews from DB (includes any previous runs)
    df_raw = load_reviews()

    if df_raw.empty:
        print('❌ No reviews found. Check the URL and try again.')
        print('   💡 Tip: Try with HEADLESS=False to see if a CAPTCHA is blocking you.')
        return pd.DataFrame()

    # Run sentiment analysis
    df = perform_sentiment_analysis(df_raw)

    # Visualise
    visualize_sentiment(df)

    # Report + save CSV
    generate_summary_report(df, save_csv=True)

    return df


# ── RUN ──
if PRODUCT_URL != "YOUR_PRODUCT_URL_HERE":
    df = run_analysis(PRODUCT_URL, max_pages=MAX_PAGES, headless=HEADLESS)

    if not df.empty:
        print(f'\n📊 df contains {len(df)} reviews. Try:')
        print('   df.head()')
        print("   df[df['sentiment'] == 'Negative']")
        print("   df['compound'].describe()")
else:
    print('⚠️  Set PRODUCT_URL above, then re-run this cell.')
    print('\nExamples:')
    print('  Amazon  : https://www.amazon.in/dp/B0XXXXXXXX')
    print('  Amazon  : https://www.amazon.in/product-reviews/B0XXXXXXXX')
    print('  Flipkart: https://www.flipkart.com/product-name/p/itemXXXXXXXX')